In [ ]:
from google.colab import drive
import os

# 1. Mount your Google Drive
drive.mount('/content/drive')

# 2. Install the latest Ultralytics package for YOLO11
!pip install -U ultralytics

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 27.2 MB/s eta 0:00:00


In [ ]:
# Create the local directory structure
!mkdir -p /content/datasets/potholes/images
!mkdir -p /content/datasets/potholes/labels

# Define your Drive paths (adjust if your folder names differ)
# Note: Ensure your Drive folders contain both .jpg and .txt files
TRAIN_DRIVE = '/content/drive/MyDrive/Street/train'
VALID_DRIVE = '/content/drive/MyDrive/Street/valid'

# Symbolically link the Drive data to the local Colab environment
if os.path.exists(TRAIN_DRIVE):
    !ln -s {TRAIN_DRIVE} /content/datasets/potholes/train
if os.path.exists(VALID_DRIVE):
    !ln -s {VALID_DRIVE} /content/datasets/potholes/valid

print("Data linked successfully.")

Data linked successfully.


In [ ]:
import yaml

data_config = {
    'train': '/content/datasets/potholes/train',
    'val': '/content/datasets/potholes/valid',
    'nc': 1,
    'names': ['pothole']
}

with open('/content/datasets/pothole_config.yaml', 'w') as f:
    yaml.dump(data_config, f)

print("Config file created at /content/datasets/pothole_config.yaml")

Config file created at /content/datasets/pothole_config.yaml


In [ ]:
from ultralytics import YOLO

# Load the YOLO11s pre-trained model
model = YOLO('yolo11s.pt')

# Start training
# We use 80 epochs for the MVP to see quick results
results = model.train(
    data='/content/datasets/pothole_config.yaml',
    epochs=80,
    imgsz=640,
    batch=16,
    name='street_mapper_mvp',
    device=0  # Uses the T4 GPU
)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.25 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/datasets/pothole_config.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=80, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=64

In [ ]:
# Load your newly trained custom model
best_model = YOLO('/content/runs/detect/street_mapper_mvp/weights/best.pt')

# Function to simulate sending location when a pothole is found
def trigger_location_alert(lat, lon, confidence):
    print(f"!!! DAMAGE DETECTED !!!")
    print(f"Location: {lat}, {lon}")
    print(f"Confidence: {confidence:.2f}")
    # In a full app, this would send an API request to your dashboard

# Run detection on a test video
# 'stream=True' is used for efficient memory handling
results = best_model.predict(source='sample_video.mp4', conf=0.5, stream=True)

for result in results:
    if len(result.boxes) > 0:
        # For the MVP, we use a placeholder for current GPS
        # In the real world, this comes from your phone's GPS sensor
        current_lat, current_lon = 22.7196, 75.8577

        conf = result.boxes.conf[0].item()
        trigger_location_alert(current_lat, current_lon, conf)


video 1/1 (frame 1/375) /content/sample_video.mp4: 384x640 9 potholes, 88.6ms
!!! DAMAGE DETECTED !!!
Location: 22.7196, 75.8577
Confidence: 0.86
video 1/1 (frame 2/375) /content/sample_video.mp4: 384x640 9 potholes, 15.7ms
!!! DAMAGE DETECTED !!!
Location: 22.7196, 75.8577
Confidence: 0.85
video 1/1 (frame 3/375) /content/sample_video.mp4: 384x640 10 potholes, 15.0ms
!!! DAMAGE DETECTED !!!
Location: 22.7196, 75.8577
Confidence: 0.81
video 1/1 (frame 4/375) /content/sample_video.mp4: 384x640 9 potholes, 12.1ms
!!! DAMAGE DETECTED !!!
Location: 22.7196, 75.8577
Confidence: 0.79
video 1/1 (frame 5/375) /content/sample_video.mp4: 384x640 10 potholes, 17.2ms
!!! DAMAGE DETECTED !!!
Location: 22.7196, 75.8577
Confidence: 0.85
video 1/1 (frame 6/375) /content/sample_video.mp4: 384x640 9 potholes, 14.3ms
!!! DAMAGE DETECTED !!!
Location: 22.7196, 75.8577
Confidence: 0.85
video 1/1 (frame 7/375) /content/sample_video.mp4: 384x640 9 potholes, 11.7ms
!!! DAMAGE DETECTED !!!
Location: 22.7196, 

In [ ]:
import csv
from ultralytics import YOLO
import datetime

# 1. Load your trained model
model = YOLO('/content/runs/detect/street_mapper_mvp/weights/best.pt')

# 2. Setup the CSV file
csv_file = 'pothole_detections.csv'
header = ['Timestamp', 'Latitude', 'Longitude', 'Issue_Type', 'Confidence']

with open(csv_file, mode='w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(header)

print(f"Created {csv_file}. Starting detection...")

# 3. Run Inference on Video
# Replace 'your_video.mp4' with your actual file path
results = model.predict(source='sample_video.mp4', conf=0.5, stream=True)

# 4. Simulated GPS Logic (For MVP)
# In a real scenario, you'd match the frame timestamp to a GPS log
base_lat, base_lon = 19.0760, 72.8777  # Example: Mumbai coordinates

for i, result in enumerate(results):
    if len(result.boxes) > 0:
        for box in result.boxes:
            conf = float(box.conf[0])
            cls = int(box.cls[0])
            label = model.names[cls]

            # Simple simulation: add a tiny bit of movement per frame
            current_lat = base_lat + (i * 0.00001)
            current_lon = base_lon + (i * 0.00001)
            timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")

            # Write to CSV
            with open(csv_file, mode='a', newline='') as f:
                writer = csv.writer(f)
                writer.writerow([timestamp, current_lat, current_lon, label, f"{conf:.2f}"])

            print(f"Saved: {label} at {current_lat}, {current_lon}")

print(f"Processing finished. You can now download {csv_file}")

Created pothole_detections.csv. Starting detection...

video 1/1 (frame 1/375) /content/sample_video.mp4: 384x640 9 potholes, 11.5ms
Saved: pothole at 19.076, 72.8777
Saved: pothole at 19.076, 72.8777
Saved: pothole at 19.076, 72.8777
Saved: pothole at 19.076, 72.8777
Saved: pothole at 19.076, 72.8777
Saved: pothole at 19.076, 72.8777
Saved: pothole at 19.076, 72.8777
Saved: pothole at 19.076, 72.8777
Saved: pothole at 19.076, 72.8777
video 1/1 (frame 2/375) /content/sample_video.mp4: 384x640 9 potholes, 11.3ms
Saved: pothole at 19.07601, 72.87771000000001
Saved: pothole at 19.07601, 72.87771000000001
Saved: pothole at 19.07601, 72.87771000000001
Saved: pothole at 19.07601, 72.87771000000001
Saved: pothole at 19.07601, 72.87771000000001
Saved: pothole at 19.07601, 72.87771000000001
Saved: pothole at 19.07601, 72.87771000000001
Saved: pothole at 19.07601, 72.87771000000001
Saved: pothole at 19.07601, 72.87771000000001
video 1/1 (frame 3/375) /content/sample_video.mp4: 384x640 10 pothole

In [ ]:
from ultralytics import YOLO

# Load your trained pothole model
model = YOLO('yolo11s.pt')

# Export to TFLite format (INT8 quantization makes it super fast on phones)
model.export(format='tflite', int8=True)

# This will create a 'best_int8.tflite' file. This is what goes on your phone.

Ultralytics 8.4.25 🚀 Python-3.12.12 torch-2.10.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
WARNING ⚠️ INT8 export requires a missing 'data' arg for calibration. Using default 'data=coco8.yaml'.
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/
YOLO11s summary (fused): 100 layers, 9,443,760 parameters, 0 gradients, 21.5 GFLOPs

PyTorch: starting from 'yolo11s.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 84, 8400) (18.4 MB)
requirements: Ultralytics requirements ['sng4onnx>=1.0.1', 'onnx_graphsurgeon>=0.3.26', 'ai-edge-litert>=1.2.0', 'onnx>=1.12.0,<2.0.0', 'onnx2tf>=1.26.3,<1.29.0', 'onnxslim>=0.1.71', 'onnxruntime-gpu'] not found, attempting AutoUpdate...
Using Python 3.12.12 environment at: /usr
Resolved 18 packages in 2.39s
Prepared 9 packages in 6.39s
Installed 9 packages in 254ms
 + ai-edge-litert==2.1.3
 + backports-strenum==1.3.1
 + colorama==0.4.6
 + onnx==1.20.1
 + onnx-

'yolo11s_saved_model/yolo11s_int8.tflite'